In [ ]:
'''
    Thermodynamic properties of NCM layered oxide from Wang-Landau simulation

    Created on Nov 28, 2022 at RISM (Shinshu University)
    Last update: Jul 30, 2026 16:30 JST

    Copyright © 2022-2026 Quang Nguyen. All rights reserved.
'''

import os
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from matplotlib import ticker

# Calculate thermodynamic variables
def thermo(T, E, lngE, N):
    factor = 96.485332123
    kB = 8.617333262 * 10 ** (-5)
    Z  = np.zeros(len(T))
    U  = np.zeros(len(T))
    F  = np.zeros(len(T))
    E2 = np.zeros(len(T))
    S  = np.zeros(len(T))
    Cv = np.zeros(len(T))
    E0    = E[0]
    lngE0 = lngE[0]
    gE    = np.exp(lngE - lngE0)
    for i in range(len(T)):
        for j in range(len(E)):
            w = np.exp(- (E[j] - E0) / (kB * T[i]))
            Z[i]  += gE[j] * w
            U[i]  += E[j] * gE[j] * w
            E2[i] += E[j] ** 2 * gE[j] * w
        U[i]  *= 1.0 / Z[i]
        E2[i] *= 1.0 / Z[i]
        F[i]   = - kB * T[i] * np.log(Z[i]) + E0
        S[i]   = (U[i] - F[i]) / T[i]
        Cv[i]  = (E2[i] - U[i] ** 2) / (kB * T[i] ** 2)
    U  *= (1 / N) * factor
    F  *= (1 / N) * factor
    S  *= (1 / N) * factor
    Cv *= (1 / N) * factor 
    return U, F, S, Cv

# Get data from file and calculate thermodynamic variables
color1, color2, color3 = '#ff7f0e', '#d62728', '#1f77b4'
label1, label2, label3 = 'NCM523', 'NCM622', 'NCM811'
datafile1           = '../MCMC_Stage1/NCM523_HnDOSvsE.dat'
datafile2           = '../MCMC_Stage1/NCM622_HnDOSvsE.dat'
datafile3           = '../MCMC_Stage1/NCM811_HnDOSvsE.dat'
data1               = np.loadtxt(datafile1, usecols=[1,5])
data2               = np.loadtxt(datafile2, usecols=[1,5])
data3               = np.loadtxt(datafile3, usecols=[1,5])
N                   = 60
eVtokJmol           = 96.485332123
E1, E2, E3          = data1[:, 0], data2[:, 0], data3[:, 0]
lngE1, lngE2, lngE3 = data1[:, 1], data2[:, 1], data3[:, 1]
T                   = np.linspace(5, 2000, 400)
U1, F1, S1, Cv1     = thermo(T, E1, lngE1, N)
U2, F2, S2, Cv2     = thermo(T, E2, lngE2, N)
U3, F3, S3, Cv3     = thermo(T, E3, lngE3, N)

# Plot normalized DOS versus normalized energy
plt.figure(figsize=(8.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%4.0f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(4.0))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(2.0))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(10.0))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(5.0))
plt.plot(E1 - np.min(E1), lngE1 - np.min(lngE1), color=color1, linestyle='solid', linewidth=2, label=label1)
plt.plot(E2 - np.min(E2), lngE2 - np.min(lngE2), color=color2, linestyle='solid', linewidth=2, label=label2)
plt.plot(E3 - np.min(E3), lngE3 - np.min(lngE3), color=color3, linestyle='solid', linewidth=2, label=label3)
plt.xlim([0.0 - 4/8, 16.0 + 6/8])
plt.ylim([0.0 - 5/4, 50.0 + 5/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$E - E_{0}$ (eV)', fontsize=26)
plt.ylabel(r'$ln(g(E))$', fontsize=26)
plt.legend(loc='upper right', frameon=False, handlelength=1.0, fontsize=20)
plt.tight_layout()
plt.show()

# Plot thermodynamic variables: Mixing entropy
plt.figure(figsize=(8.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%4.1f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.4))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.2))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(1.0))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.5))
plt.plot(T / 1000, S1 * 1000, color=color1, linestyle='solid', linewidth=2, label=label1)
plt.plot(T / 1000, S2 * 1000, color=color2, linestyle='solid', linewidth=2, label=label2)
plt.plot(T / 1000, S3 * 1000, color=color3, linestyle='solid', linewidth=2, label=label3)
plt.xlim([0.0 - 2/32, 2.0 + 2/32])
plt.ylim([0.0 - 0.5/4, 5.0 + 0.5/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$S$ (J/mol/K)', fontsize=26)
plt.legend(loc='upper left', frameon=False, handlelength=1.0, fontsize=20)
plt.tight_layout()
plt.show()

# Plot thermodynamic variables: Specific heat
plt.figure(figsize=(8.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%4.1f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.4))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.2))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(1.0))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.5))
plt.plot(T / 1000, Cv1 * 1000, color=color1, linestyle='solid', linewidth=2, label=label1)
plt.plot(T / 1000, Cv2 * 1000, color=color2, linestyle='solid', linewidth=2, label=label2)
plt.plot(T / 1000, Cv3 * 1000, color=color3, linestyle='solid', linewidth=2, label=label3)
plt.xlim([0.0 - 2/32, 2.0 + 2/32])
plt.ylim([0.0 - 0.5/4, 5.0 + 0.5/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$C_{v}$ (J/mol/K)', fontsize=26)
plt.legend(loc='upper right', frameon=False, handlelength=1.0, fontsize=20, ncols=3, columnspacing=1.0)
plt.tight_layout()
plt.show()

# Plot thermodynamic variables: Internal energy
plt.figure(figsize=(8.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%4.1f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.4))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.2))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(1.0))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.5))
plt.plot(T / 1000, U1 - np.min(U1), color=color1, linestyle='solid', linewidth=2, label=label1)
plt.plot(T / 1000, U2 - np.min(U2), color=color2, linestyle='solid', linewidth=2, label=label2)
plt.plot(T / 1000, U3 - np.min(U3), color=color3, linestyle='solid', linewidth=2, label=label3)
plt.xlim([0.0 - 2/32, 2.0 + 2/32])
plt.ylim([0.0 - 0.5/4, 5.0 + 0.5/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$U - U_{0}$ (kJ/mol)', fontsize=26)
plt.legend(loc='upper left', frameon=False, handlelength=1.0, fontsize=20)
plt.tight_layout()
plt.show()

# Plot thermodynamic variables: Free energy
plt.figure(figsize=(8.4, 5.6))
plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))
plt.gca().yaxis.set_major_formatter(ticker.FormatStrFormatter('%4.1f'))
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(0.4))
plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.2))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(1.6))
plt.gca().yaxis.set_minor_locator(ticker.MultipleLocator(0.8))
plt.plot(T / 1000, F1 - np.max(F1), color=color1, linestyle='solid', linewidth=2, label=label1)
plt.plot(T / 1000, F2 - np.max(F2), color=color2, linestyle='solid', linewidth=2, label=label2)
plt.plot(T / 1000, F3 - np.max(F3), color=color3, linestyle='solid', linewidth=2, label=label3)
plt.xlim([0.0 - 2/32, 2.0 + 2/32])
plt.ylim([-8.0 - 0.8/4, 0.0 + 0.8/4])
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.xlabel(r'$1000 × T$ (K)', fontsize=26)
plt.ylabel(r'$F - F_{0}$ (kJ/mol)', fontsize=26)
plt.legend(loc='lower left', frameon=False, handlelength=1.0, fontsize=20)
plt.tight_layout()
plt.show()